## Brain Map Visualization — Subcortical Volume (Prodromal Subgroups)

Companion to `ggseg_volume_subcortical.ipynb`. Compares **RBD** and **Hyposmia**
prodromal subgroups against **Healthy Controls** using the FreeSurfer `aseg` atlas.

**Contrasts:** RBD vs HC · Hyposmia vs HC

**Group N:** RBD = 119 · Hyposmia = 160 · HC = 192

**Input files:**
- `results/rbd_vs_hc_parcelwise_ttest_volume_subcortical.csv`
- `results/hyposmia_vs_hc_parcelwise_ttest_volume_subcortical.csv`

**Output dir:** `results/figures/ggseg/subgroups/subcortical/`

**Figures produced:**
- Per-contrast full + FDR-masked maps
- `combined_paper_style_unthresh_vs_fdr.png` — unthresholded |g| vs FDR-corrected (2 × 2 layout)

In [ ]:
# install.packages(c("tidyverse", "patchwork", "remotes"))
# remotes::install_github("LCBC-UiO/ggseg")
suppressPackageStartupMessages({
  library(ggseg)
  library(tidyverse)
  library(patchwork)
})
cat("ggseg:", as.character(packageVersion("ggseg")), "\n")

In [ ]:
RESULTS_DIR <- "../../results"
FIG_DIR     <- file.path(RESULTS_DIR, "figures", "ggseg", "subgroups", "subcortical")
dir.create(FIG_DIR, recursive = TRUE, showWarnings = FALSE)

GROUP_N <- list(
  "RBD vs HC"      = c(n1 = 119, n2 = 192),
  "Hyposmia vs HC" = c(n1 = 160, n2 = 192)
)

TTEST_FILES <- list(
  "RBD vs HC"      = file.path(RESULTS_DIR, "rbd_vs_hc_parcelwise_ttest_volume_subcortical.csv"),
  "Hyposmia vs HC" = file.path(RESULTS_DIR, "hyposmia_vs_hc_parcelwise_ttest_volume_subcortical.csv")
)

df_sc <- imap_dfr(TTEST_FILES, function(path, cname) {
  read_csv(path, show_col_types = FALSE) %>% mutate(contrast = cname)
}) %>%
  filter(!is.na(pvalue))

cat("Loaded", nrow(df_sc), "parcel x contrast rows\n")
cat("Parcels per contrast:", n_distinct(df_sc$parcel), "\n")

In [ ]:
contrasts <- setNames(as.list(names(GROUP_N)), names(GROUP_N))

hedges_g_from_t <- function(t, n1, n2, df_resid) {
  -t * sqrt(1/n1 + 1/n2) * (1 - 3 / (4 * df_resid - 1))
}

df_sc <- df_sc %>%
  rowwise() %>%
  mutate(g = hedges_g_from_t(
    Tvalue,
    GROUP_N[[contrast]][["n1"]],
    GROUP_N[[contrast]][["n2"]],
    df
  )) %>%
  ungroup()

cat("Hedges' g range:", round(range(df_sc$g, na.rm = TRUE), 3), "\n")

In [ ]:
df_sc <- df_sc %>%
  mutate(
    hemi      = if_else(str_detect(parcel, "hemi-L"), "left", "right"),
    raw_label = str_replace(parcel, ".*_lab-", ""),
    label     = paste0(
      if_else(hemi == "left", "Left", "Right"), "-",
      str_replace(raw_label, "\\+", "-")
    ),
    region    = str_replace(raw_label, "\\+area", " area")
  )

atlas_labels <- as.data.frame(aseg())$label
bad <- df_sc %>% filter(!label %in% atlas_labels) %>% pull(label) %>% unique()
if (length(bad) == 0) {
  cat("All", n_distinct(df_sc$label), "labels match the aseg atlas.\n")
} else {
  cat("WARNING — unmatched labels:", bad, "\n")
}

In [ ]:
df_sc <- df_sc %>%
  group_by(contrast) %>%
  mutate(p_fdr = p.adjust(pvalue, method = "fdr")) %>%
  ungroup()

df_sc %>%
  group_by(contrast) %>%
  summarise(
    n_sig_fdr = sum(p_fdr < 0.05, na.rm = TRUE),
    n_sig_unc = sum(pvalue < 0.05, na.rm = TRUE),
    n         = n(),
    g_max_abs = round(max(abs(g), na.rm = TRUE), 3)
  )

In [ ]:
G_LIMIT <- 0.8

scale_g <- scale_fill_gradient2(
  low      = "#2166AC",
  mid      = "white",
  high     = "#D6604D",
  midpoint = 0,
  limits   = c(-G_LIMIT, G_LIMIT),
  oob      = scales::squish,
  name     = "Hedges' g",
  na.value = "grey85"
)

G_LIMIT_PAPER <- ceiling(max(abs(df_sc$g), na.rm = TRUE) * 10) / 10
scale_g_paper <- scale_fill_gradient(
  low      = "white",
  high     = "#4393C3",
  limits   = c(0, G_LIMIT_PAPER),
  oob      = scales::squish,
  name     = "|Hedges' g|",
  na.value = "#f5f5f5"
)

cat("Paper scale upper limit:", G_LIMIT_PAPER, "\n")

# Helper: join data to aseg atlas and filter to coronal slice
coronal_join <- function(df) {
  brain_join(df, aseg()) %>% dplyr::filter(view == "coronal_1")
}

### Full Hedges' g maps + FDR-masked maps (per contrast, coronal view)

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 3.5)

for (cname in names(contrasts)) {
  d <- df_sc %>% filter(contrast == cname)

  p_full <- ggplot(coronal_join(d %>% select(label, g))) +
    geom_sf(aes(fill = g), colour = "white") +
    scale_g +
    labs(title = cname,
         subtitle = "Subcortical volume — Hedges' g (all structures; eTIV-corrected)") +
    theme_void() +
    theme(plot.title      = element_text(hjust = 0.5, size = 13, face = "bold"),
          plot.subtitle   = element_text(hjust = 0.5, size = 10, colour = "grey40"),
          legend.position = "right")

  p_mask <- ggplot(
    coronal_join(d %>% mutate(g_sig = if_else(p_fdr < 0.05, g, NA_real_)) %>% select(label, g_sig))
  ) +
    geom_sf(aes(fill = g_sig), colour = "white") +
    scale_g +
    labs(title = cname,
         subtitle = "Subcortical volume — FDR-masked (q < 0.05; eTIV-corrected)") +
    theme_void() +
    theme(plot.title      = element_text(hjust = 0.5, size = 13, face = "bold"),
          plot.subtitle   = element_text(hjust = 0.5, size = 10, colour = "grey40"),
          legend.position = "right")

  combined <- p_full + p_mask + plot_layout(guides = "collect") & theme(legend.position = "right")
  print(combined)

  fname <- tolower(str_replace_all(cname, " ", "_"))
  ggsave(file.path(FIG_DIR, paste0(fname, ".png")), combined, width = 10, height = 3.5, dpi = 300)
  cat("Saved:", fname, "\n")
}

### Combined paper-style figure: unthresholded vs FDR-corrected (2 × 2 layout)

| | Left (unthresholded |g|) | Right (FDR q < 0.05) |
|---|---|---|
| **Row 1 (a, b)** | RBD vs HC | RBD vs HC |
| **Row 2 (c, d)** | Hyposmia vs HC | Hyposmia vs HC |

In [ ]:
FIG_DIR_COMBINED <- file.path(FIG_DIR, "combined")
dir.create(FIG_DIR_COMBINED, recursive = TRUE, showWarnings = FALSE)

panel_theme <- theme(
  plot.title    = element_text(hjust = 0.5, size = 9.5, face = "bold",
                               margin = margin(t = 10, b = 3)),
  plot.subtitle = element_text(hjust = 0.5, size = 7.5, colour = "grey35",
                               margin = margin(b = 10)),
  plot.margin   = margin(t = 8, r = 6, b = 12, l = 6)
)

plots_pap_unc <- list()
plots_pap_fdr <- list()

for (cname in names(contrasts)) {
  d     <- df_sc %>% filter(contrast == cname)
  n_sig <- sum(d$p_fdr < 0.05, na.rm = TRUE)
  sig_label <- if (n_sig == 0) "none" else {
    d %>% filter(p_fdr < 0.05) %>% pull(region) %>% unique() %>% paste(collapse = ", ")
  }

  plots_pap_unc[[cname]] <- ggplot(
    coronal_join(d %>% mutate(g_abs = abs(g)) %>% select(label, g_abs))
  ) +
    geom_sf(aes(fill = g_abs), colour = "grey60") +
    scale_g_paper +
    labs(title = cname, subtitle = "|Hedges' g| (all structures, unthresholded)") +
    theme_void() + panel_theme

  plots_pap_fdr[[cname]] <- ggplot(
    coronal_join(
      d %>% mutate(g_abs = if_else(p_fdr < 0.05, abs(g), NA_real_)) %>% select(label, g_abs)
    )
  ) +
    geom_sf(aes(fill = g_abs), colour = "grey60") +
    scale_g_paper +
    labs(title = cname,
         subtitle = paste0("FDR q < 0.05: n = ", n_sig, " (", sig_label, ")")) +
    theme_void() + panel_theme
}

fig_combined <- (
  (plots_pap_unc[[1]] | plots_pap_fdr[[1]]) /
  (plots_pap_unc[[2]] | plots_pap_fdr[[2]])
) +
  plot_layout(guides = "collect") +
  plot_annotation(
    title      = "Subcortical Volume (Unthresholded vs FDR-corrected) — Prodromal Subgroups",
    subtitle   = "Left (a, c): all |Hedges' g| | Right (b, d): FDR-corrected (q < 0.05)",
    tag_levels = "a",
    theme = theme(
      plot.title    = element_text(hjust = 0.5, size = 13, face = "bold"),
      plot.subtitle = element_text(hjust = 0.5, size = 9,  colour = "grey40")
    )
  ) &
  theme(legend.position = "right")

options(repr.plot.width = 10, repr.plot.height = 8)
print(fig_combined)

out_path <- file.path(FIG_DIR_COMBINED, "combined_paper_style_unthresh_vs_fdr.png")
ggsave(out_path, fig_combined, width = 10, height = 8, dpi = 300)
cat("Saved:", out_path, "\n")

### Diverging pastel Hedges’ g maps — subgroup subcortical volume

**Figure.** Diverging Hedges’ g maps of subcortical volume for prodromal subgroups (RBD vs HC; Hyposmia vs HC). Light pink = volume reduction (g < 0); light blue = volume increase (g > 0); grey = no data. Scale capped at ±0.30. Left column: all structures unthresholded; right column: FDR q < 0.05 parcels only. No FDR-significant subcortical differences were observed in either subgroup.

In [ ]:
CNAMES_DISPLAY_SUB_SC <- c(
  "RBD vs HC"      = "RBD vs HC",
  "Hyposmia vs HC" = "hyposmia vs HC"
)

G_LIMIT_PASTEL <- 0.30

scale_diverging_pastel_sc <- scale_fill_gradient2(
  low      = "lightpink",
  mid      = "white",
  high     = "#AED6F1",
  midpoint = 0,
  limits   = c(-G_LIMIT_PASTEL, G_LIMIT_PASTEL),
  oob      = scales::squish,
  name     = "Hedges' g",
  na.value = "grey85"
)

plots_div_fdr_sub <- list()
plots_div_unc_sub <- list()

for (cname in names(contrasts)) {
  d     <- df_sc %>% filter(contrast == cname)

  plots_div_fdr_sub[[cname]] <- ggplot(
    coronal_join(
      d %>% mutate(g_plot = if_else(p_fdr < 0.05, g, NA_real_)) %>% select(label, g_plot)
    )
  ) +
    geom_sf(aes(fill = g_plot), colour = "grey60") +
    scale_diverging_pastel_sc +
    labs(title = CNAMES_DISPLAY_SUB_SC[[cname]]) +
    theme_void() + panel_theme

  plots_div_unc_sub[[cname]] <- ggplot(
    coronal_join(d %>% mutate(g_plot = g) %>% select(label, g_plot))
  ) +
    geom_sf(aes(fill = g_plot), colour = "grey60") +
    scale_diverging_pastel_sc +
    labs(
      title    = CNAMES_DISPLAY_SUB_SC[[cname]],
    ) +
    theme_void() + panel_theme
}

options(repr.plot.width = 10, repr.plot.height = 10)

fig_div_sub_sc <- (
  (plots_div_unc_sub[[1]] | plots_div_fdr_sub[[1]]) /
  (plots_div_unc_sub[[2]] | plots_div_fdr_sub[[2]])
) +
  plot_layout(guides = "collect") +
  plot_annotation(
    title    = "Subcortical Volume Subgroups: Diverging Hedges' g",
    subtitle = "Left: all structures, unthresholded | Right: FDR q < 0.05 only\nLight pink = volume loss (g < 0) | Light blue = volume gain (g > 0)",
    tag_levels = "a",
    theme = theme(
      plot.title    = element_text(hjust = 0.5, size = 13, face = "bold"),
      plot.subtitle = element_text(hjust = 0.5, size = 9,  colour = "grey40")
    )
  ) &
  theme(legend.position = "right")

print(fig_div_sub_sc)

out_path_sub_sc <- file.path(FIG_DIR, "figure_diverging_subcortical_subgroups_combined.png")
ggsave(out_path_sub_sc, fig_div_sub_sc, width = 10, height = 10, dpi = 300)
cat("Saved:", out_path_sub_sc, "\n")

### Results table

In [ ]:
df_sc %>%
  select(contrast, hemi, region, g, Tvalue, pvalue, p_fdr) %>%
  mutate(
    across(where(is.numeric), \(x) round(x, 4)),
    sig = case_when(p_fdr < 0.001 ~ "***", p_fdr < 0.01 ~ "**",
                    p_fdr < 0.05  ~ "*",   TRUE ~ "")
  ) %>%
  arrange(contrast, desc(abs(g)))